In [1]:
import pytesseract
from pytesseract import Output
from PIL import Image
from transformers import LayoutLMv3Tokenizer, LayoutLMv3ForTokenClassification, LayoutLMv3ImageProcessor
import torch

In [2]:
# Load the image
image_path = "Screenshot1.png"  # Replace with your image path
image = Image.open(image_path).convert("RGB")
image

FileNotFoundError: [Errno 2] No such file or directory: 'Screenshot1.png'

In [3]:
import pytesseract
from pytesseract import Output
from PIL import Image
from transformers import LayoutLMv3Tokenizer, LayoutLMv3ForTokenClassification, LayoutLMv3ImageProcessor
import torch

# Load the image
image_path = "Screenshot.png"  # Replace with your image path
image = Image.open(image_path).convert("RGB")

# Perform OCR to extract words and bounding boxes
ocr_data = pytesseract.image_to_data(image, output_type=Output.DICT)

words = []
bboxes = []

width, height = image.size

for i in range(len(ocr_data['text'])):
    if int(ocr_data['conf'][i]) > 0 and ocr_data['text'][i].strip() != "":
        words.append(ocr_data['text'][i])
        x, y, w, h = ocr_data['left'][i], ocr_data['top'][i], ocr_data['width'][i], ocr_data['height'][i]
        # Normalize to 0–1000 as expected by LayoutLMv3
        bbox = [
            int(1000 * x / width),
            int(1000 * y / height),
            int(1000 * (x + w) / width),
            int(1000 * (y + h) / height)
        ]
        bboxes.append(bbox)

print("Extracted words:", words)
print("Corresponding bounding boxes:", bboxes)


Extracted words: ['Clicks', 'by', 'CTAs', 'General', 'avail', 'messaging', 'promotes', 'Research', 'and', 'Choosing,', 'while', 'Upgrade', 'messaging', 'drives', 'more', 'down-funnel', 'Shopping', 'and', 'Affordability', 'clicks', 'on', 'Series', '10', '“Buy”,', 'Trade-in', 'and', 'Retail', 'Value', 'Props.', 'Control', 'General', 'Avail', 'Messaging', 'Variation', 'Upgrade', 'Messaging', '%', 'Difference', 'Variation', 'versus', 'Control', 'Apple', 'Watch', 'Series', '10', 'Apple', 'Watch', 'Ultra', '2', 'Learn', 'more', 'See', 'what', 'your', 'device', 'is', 'worth', '>', 'settee', 'Apple', 'Watch', 'Studio', 'Create', 'your', 'style', '>', 'Compare', 'Compare', 'all', 'models', '>', 'Retail', 'Value', 'Props', '(Apple', 'Card)', 'Learn', 'more', '>', 'Shop', 'online', '>', 'Retail', 'Value', 'Props', 'Find', 'a', 'store', '>', 'Footer', 'Links', 'Find', 'a', 'Store', 'Get', 'the', 'Apple', 'Store', 'App', 'a', 'a', 'ES', '7', 'sex', 'esx', 'Jo.5k', 'joek', '0.2K', 'nae', 'sun', '0.1

In [4]:
# Load tokenizer and image processor
model_name = "microsoft/layoutlmv3-base"
tokenizer = LayoutLMv3Tokenizer.from_pretrained(model_name)
processor = LayoutLMv3ImageProcessor.from_pretrained(model_name)

# Tokenize the text
encoding = tokenizer(
    words,
    boxes=bboxes,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=512
)

# Process image for pixel values
pixel_values = processor(images=image, return_tensors="pt").pixel_values


In [5]:
# Load model
model = LayoutLMv3ForTokenClassification.from_pretrained(model_name)

# Combine inputs
inputs = {
    "input_ids": encoding["input_ids"],
    "attention_mask": encoding["attention_mask"],
    "bbox": encoding["bbox"],
    "pixel_values": pixel_values
}

# Run through model
with torch.no_grad():
    outputs = model(**inputs)

# Output logits
logits = outputs.logits
predictions = torch.argmax(logits, dim=-1)

# Print predictions (token-level)
print("Predicted token classes:", predictions)


Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/opt/anaconda3/envs/mds/lib/python3.12/site-packages/transformers/modeling_utils.py:1575: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Predicted token classes: tensor([[1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
   

In [6]:
from PIL import Image
import easyocr
import numpy as np

# Load your image using PIL
image = Image.open("Screenshot.png")  # or any other source
image_np = np.array(image)  # Convert PIL image to NumPy array

# Initialize EasyOCR reader
reader = easyocr.Reader(['en'])

# Run OCR on the NumPy image
results = reader.readtext(image_np)


In [7]:
words = []
bboxes = []

for (bbox, text, conf) in results:
    words.append(text)
    # bbox is a list of 4 (x, y) tuples
    x_coords = [pt[0] for pt in bbox]
    y_coords = [pt[1] for pt in bbox]
    x0, y0, x1, y1 = min(x_coords), min(y_coords), max(x_coords), max(y_coords)
    # Normalize to LayoutLM format (0-1000)
    box = [
        int(x0 / width * 1000),
        int(y0 / height * 1000),
        int(x1 / width * 1000),
        int(y1 / height * 1000),
    ]
    bboxes.append(box)


In [8]:
from PIL import ImageDraw, ImageFont

# Create a copy of the image to draw on
drawn_image = image.copy()
draw = ImageDraw.Draw(drawn_image)

# Optional: Set a font
try:
    font = ImageFont.truetype("arial.ttf", size=14)
except:
    font = ImageFont.load_default()

# Draw each bounding box with text
for word, box in zip(words, bboxes):
    # Rescale box back to image size
    x0 = int(box[0] * width / 1000)
    y0 = int(box[1] * height / 1000)
    x1 = int(box[2] * width / 1000)
    y1 = int(box[3] * height / 1000)

    draw.rectangle(((x0, y0), (x1, y1)), outline="red", width=2)
    draw.text((x0, y0 - 10), word, fill="blue", font=font)

# Show image (or save it)
drawn_image.show()
# drawn_image.save("image_with_boxes.jpg")


In [9]:
from sklearn.metrics.pairwise import cosine_similarity
import torch
from PIL import Image
from transformers import LayoutLMv3Tokenizer, LayoutLMv3ForTokenClassification, LayoutLMv3ImageProcessor

# Load the image
image_path = "Screenshot.png"  # Replace with your image path
image = Image.open(image_path).convert("RGB")

In [10]:
# Tokenize the text and prepare inputs
encoding = tokenizer(
    words,
    boxes=bboxes,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=512
)

# Process image for pixel values
pixel_values = processor(images=image, return_tensors="pt").pixel_values
pixel_values

tensor([[[[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]],

         [[1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          ...,
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.],
          [1., 1., 1.,  ..., 1., 1., 1.]]]])

In [11]:
# Load the model with output_hidden_states=True to get embeddings
model = LayoutLMv3ForTokenClassification.from_pretrained(model_name, output_hidden_states=True)

Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
# Combine inputs
inputs = {
    "input_ids": encoding["input_ids"],
    "attention_mask": encoding["attention_mask"],
    "bbox": encoding["bbox"],
    "pixel_values": pixel_values
}

In [13]:
# Run through the model to get embeddings
with torch.no_grad():
    outputs = model(**inputs)

# Now, extract the token-level embeddings from hidden states
token_embeddings = outputs.hidden_states[-1]  # Last hidden state
document_embedding = torch.mean(token_embeddings, dim=1).squeeze().cpu().numpy()
token_embeddings

/opt/anaconda3/envs/mds/lib/python3.12/site-packages/transformers/modeling_utils.py:1575: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


tensor([[[ 0.1056,  0.1261,  0.0089,  ..., -0.1857, -0.1271,  0.0727],
         [-0.0885, -0.3766, -0.0225,  ...,  0.4976, -0.4153,  0.3320],
         [ 0.1203,  0.0150,  0.0980,  ...,  0.5061, -0.2430,  0.3499],
         ...,
         [ 0.1286,  0.2276,  0.0084,  ..., -0.4518,  0.2071,  0.1709],
         [-0.0205,  0.3090,  0.2076,  ..., -1.5625,  0.4047,  0.0095],
         [ 0.1800, -0.0862,  0.0491,  ..., -0.9321,  0.0934, -0.0943]]])

In [14]:
# Query text
query_text = "For Trade-in and Retail Value Props what is the difference between Control and Variation for Apple Watch Studio."

query_text = "What is Trump Tarrif On China."

# Dummy bounding boxes (just as placeholders, as we are not processing actual layout)
dummy_bboxes = [[0, 0, 1000, 1000]] * len(query_text.split())

# Tokenize the query text with dummy bounding boxes
query_encoding = tokenizer(
    query_text.split(),  # Tokenize the query into words
    boxes=dummy_bboxes,  # Use dummy bounding boxes
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=512
)

# Run the model
with torch.no_grad():
    query_outputs = model(**query_encoding)

# Extract query embeddings (mean of last hidden state)
query_embedding = torch.mean(query_outputs.hidden_states[-1], dim=1).squeeze().cpu().numpy()

In [16]:
# Calculate cosine similarity
similarity_score = cosine_similarity([document_embedding], [query_embedding])

print(f"Similarity score between the query and document: {similarity_score[0][0]}")

Similarity score between the query and document: 0.834879457950592
